# 🔬 TraceScope AI 2.0 — Google Colab Pro Master Execution Pipeline
### Persistent Google Drive Workspace: `TraceScoop_2_codes` (A100 / L4 GPU)

> **Repository:** [SWAPNILSHAW/TraceScope_2](https://github.com/SWAPNILSHAW/TraceScope_2.git)  
> **Google Drive Folder:** `My Drive/TraceScoop_2_codes/TraceScope_2`  
> **Target System:** TraceScope AI 2.0 (Scanner Source Attribution + Document Forensics)  
> **Supported Phases:** Phase 5 (ResNet-18), Phase 6 (Hybrid CNN), Phase 7 (Ablations), Phase 8 (Robustness)  

---

## ⚙️ Step 0: Check GPU & Hardware Acceleration

In [ ]:
!nvidia-smi
import torch
import tensorflow as tf
print("PyTorch CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
print("TensorFlow GPUs:", tf.config.list_physical_devices('GPU'))

## 📁 Step 1: Mount Google Drive & Setup `TraceScoop_2_codes` Folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_WORKSPACE = '/content/drive/MyDrive/TraceScoop_2_codes'
os.makedirs(DRIVE_WORKSPACE, exist_ok=True)

%cd $DRIVE_WORKSPACE
if not os.path.exists('TraceScope_2'):
    !git clone https://github.com/SWAPNILSHAW/TraceScope_2.git
%cd TraceScope_2
!git pull origin main
print(" Workspace ready in Google Drive:", os.getcwd())

## 📦 Step 2: Install Required Dependencies

In [ ]:
!pip install -q scikit-image tifffile PyWavelets seaborn gdown

## 💾 Step 3: Verify Pre-computed Residual Cache (1.19 GB)
Ensures `results/hybrid_cnn/official_wiki_residuals.pkl` is in place.

In [ ]:
import os
import shutil

target_dir = os.path.abspath('results/hybrid_cnn')
os.makedirs(target_dir, exist_ok=True)
target_file = os.path.join(target_dir, 'official_wiki_residuals.pkl')

if os.path.exists(target_file):
    print(f" Residual cache already present: {target_file}")
    print(f" File Size: {os.path.getsize(target_file) / (1024**3):.2f} GB")
    print('Residual cache status: READY ✅')
else:
    print('🔍 Searching your Google Drive for official_wiki_residuals.pkl...')
    found_path = None
    drive_root = '/content/drive/MyDrive'
    if os.path.exists(drive_root):
        for root, dirs, files in os.walk(drive_root):
            if 'official_wiki_residuals.pkl' in files:
                found_path = os.path.join(root, 'official_wiki_residuals.pkl')
                print(f' Found in Drive: {found_path}')
                print(f' Copying into workspace ({target_file}) ...')
                shutil.copy2(found_path, target_file)
                break

    if os.path.exists(target_file):
        print(f' Residual cache loaded successfully: {os.path.getsize(target_file) / (1024**3):.2f} GB')
        print('Residual cache status: READY ✅')
    else:
        print('\n' + '='*65)
        print("❌ 'official_wiki_residuals.pkl' not yet found in Drive.")
        print('='*65)
        print('Quick Fix (Takes 1-2 minutes):')
        print('1. Open https://drive.google.com in your web browser.')
        print('2. Open folder: TraceScoop_2_codes > TraceScope_2 > results > hybrid_cnn')
        print('3. Drag and drop official_wiki_residuals.pkl from your PC:')
        print(r'   E:\infosy internship\results\hybrid_cnn\official_wiki_residuals.pkl')
        print('4. Once uploaded, re-run this cell!')
        print('='*65)
        print('Residual cache status: NOT FOUND ❌')


## 🚀 Step 4: Phase 5 — Train Deep CNN Baseline (ResNet-18 on GPU)
- Backbone: ResNet-18 initialized from scratch (no natural image bias)
- Forensic Filter: Fixed $5 \times 5$ Kraetzer-Vogler High-Pass Kernel
- Splits: `splits/train_manifest.csv` $\to$ Evaluated on locked `splits/test_manifest.csv`
- Artifacts saved directly to your Drive: `models/cnn/resnet18_best.pth`, `results/cnn/test_metrics.csv`

In [ ]:
# Train PyTorch ResNet-18 with Kraetzer-Vogler High-Pass Filter on Colab A100 GPU
!python src/cnn_model/train.py --epochs 25 --batch_size 64 --lr 0.0005

## 🚀 Step 5: Phase 6 — Train Hybrid CNN Primary Model (Dual-Branch Fusion)
- Branch A: $256 \times 256$ Residual CNN (Conv2D 32-64-128 + GAP)
- Branch B: 44 Handcrafted Forensic Features (PRNU + FFT + LBP + Texture)
- Artifacts saved directly to your Drive: `models/hybrid_cnn/scanner_hybrid.keras`

In [ ]:
# Train TensorFlow/Keras Hybrid CNN on Colab GPU
!python src/hybrid_cnn/train_hybrid_cnn.py

## 💾 Step 6: Confirmation of Drive Persistence

In [ ]:
# All models, plots, and CSV metrics are saved directly in your Google Drive:
!ls -lh models/cnn/
!ls -lh results/cnn/
print("\n All files are permanently saved in Google Drive at: /content/drive/MyDrive/TraceScoop_2_codes/TraceScope_2")